# 02 - Modelado y Entrenamiento del Modelo CMM
**Objetivo:** Cargar el dataset procesado, integrar nuevos samples, entrenar un modelo de clasificación de niveles CMM de alto rendimiento y guardar el nuevo modelo versionado.

**Modelo:** MLPClassifier (Red Neuronal) para lograr ~100% de accuracy en CIFAR-10.
**Nota:** No realizar predicciones finales en este cuaderno.

In [15]:
# Imports y preparación de carpetas
import os
import glob
import time
import joblib
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, classification_report
os.makedirs('models', exist_ok=True)
print('Librerías cargadas')

Librerías cargadas


## Carga del dataset procesado y nuevos samples
Cargamos `data/historical.parquet` y `data/new_samples.parquet` generados en el cuaderno 1. Si no existen, intentar cargar `data/processed_dataset.parquet`.

In [ ]:
# Cargar datos procesados (NPZ) generados por el cuaderno 01
if os.path.exists('data/historical.npz') and os.path.exists('data/new_samples.npz'):
    hist = np.load('data/historical.npz')
    new = np.load('data/new_samples.npz')
    X_hist, y_hist = hist['X'], hist['y']
    X_new, y_new = new['X'], new['y']
    print('Loaded historical and new_samples')
elif os.path.exists('data/processed_cifar10.npz'):
    arr = np.load('data/processed_cifar10.npz')
    X_all_tmp, y_all_tmp = arr['X'], arr['y']
    X_hist, X_new, y_hist, y_new = train_test_split(
        X_all_tmp, y_all_tmp, test_size=0.2, random_state=42, stratify=y_all_tmp
    )
    print('Loaded processed_cifar10 and split into historical/new_samples')
else:
    raise FileNotFoundError('No se encuentra dataset procesado. Ejecute el cuaderno 01 primero.')

X_all = np.concatenate([X_hist, X_new], axis=0)
y_all = np.concatenate([y_hist, y_new], axis=0)
print('Historical:', X_hist.shape, y_hist.shape, 'New_samples:', X_new.shape, y_new.shape)
print('Dataset combinado:', X_all.shape, y_all.shape, 'Clases:', np.unique(y_all).size)

Loaded historical and new_samples
Historical: (48000, 3072) (48000,) New_samples: (12000, 3072) (12000,)


In [ ]:
# Separar conjunto de entrenamiento y validación (estratificado)
X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)
classes = np.unique(y_all)
print('Train:', X_train.shape, y_train.shape)
print('Val  :', X_val.shape, y_val.shape)
print('Clases:', classes)


## Carga de un modelo previamente entrenado (si existe)
Buscamos el modelo más reciente en `models/`. Si no hay ninguno, inicializamos un `MLPClassifier` (red neuronal) para alto rendimiento.

**Justificación del MLPClassifier:**
- Mejor capacidad de aprendizaje en datos complejos como CIFAR-10 (imágenes)
- Capas ocultas permiten capturar patrones no lineales

- Objetivo: alcanzar ~100% de accuracy en entrenamiento- Se entrena con partial_fit para permitir entrenamiento incremental

In [ ]:
# Inicializar/cargar modelo y entrenar con búsqueda del mejor checkpoint en validación
model_files = glob.glob('models/model_*.joblib')
model_files = [p for p in model_files if '_meta' not in os.path.basename(p)]
model_files = sorted(model_files, key=os.path.getmtime)

if model_files:
    model_path = model_files[-1]
    model = joblib.load(model_path)
    print(f'Modelo cargado desde {model_path}')
else:
    model = MLPClassifier(
        hidden_layer_sizes=(1024, 512, 256),
        activation='relu',
        solver='adam',
        learning_rate='adaptive',
        learning_rate_init=5e-4,
        alpha=1e-4,
        batch_size=256,
        max_iter=1,
        warm_start=True,
        random_state=42,
        verbose=False,
    )
    print('No se encontró modelo previo. Inicializado MLPClassifier (1024, 512, 256)')

epochs = 15
patience = 3
best_val_acc = -np.inf
best_bytes = None
best_y_val_pred = None
history = []
no_improve = 0

for epoch in range(epochs):
    if hasattr(model, 'classes_'):
        model.partial_fit(X_train, y_train)
    else:
        model.partial_fit(X_train, y_train, classes=classes)

    y_tr_pred = model.predict(X_train)
    y_val_pred = model.predict(X_val)
    tr_acc = accuracy_score(y_train, y_tr_pred)
    val_acc = accuracy_score(y_val, y_val_pred)
    history.append({'epoch': epoch + 1, 'train_acc': float(tr_acc), 'val_acc': float(val_acc)})
    print(f'Epoch {epoch+1:02d}/{epochs} - train_acc={tr_acc:.4f} - val_acc={val_acc:.4f}')

    if val_acc > best_val_acc + 1e-4:
        best_val_acc = val_acc
        best_bytes = joblib.dumps(model)
        best_y_val_pred = y_val_pred.copy()
        no_improve = 0
    else:
        no_improve += 1

    if no_improve >= patience:
        print('Early stopping por falta de mejora en validación')
        break

if best_bytes is not None:
    model = joblib.loads(best_bytes)
    y_val_pred = best_y_val_pred
    print(f'Modelo restaurado al mejor checkpoint (val_acc={best_val_acc:.4f})')
else:
    y_val_pred = model.predict(X_val)
    best_val_acc = accuracy_score(y_val, y_val_pred)

# Guardar modelo y metadatos versionados
ts = int(time.time())
model_path = f'models/model_{ts}.joblib'
joblib.dump(model, model_path)
meta = {
    'version': ts,
    'trainer': 'MLPClassifier',
    'epochs_trained': len(history),
    'best_val_acc': float(best_val_acc),
    'history': history,
    'classes': classes.tolist(),
    'timestamp': ts,
}
joblib.dump(meta, f'models/model_{ts}_meta.joblib')
print('Modelo guardado en', model_path)


IndentationError: unexpected indent (4099498511.py, line 23)

## Importancia del versionado de modelos (breve reflexión académica)
- Permite reproducibilidad y auditoría de decisiones.
- Facilita rollbacks si una versión empeora el rendimiento.
- Permite comparar experimentos y su impacto en la toma de decisiones organizacionales.

## Métricas finales (resumen)
Mostramos un reporte de clasificación en el conjunto de validación para evaluación académica.

In [ ]:
print('Validation accuracy:', accuracy_score(y_val, y_val_pred))
print('Validation classification report:')
print(classification_report(y_val, y_val_pred, digits=4, zero_division=0))


Validation classification report:
              precision    recall  f1-score   support

           0     0.2396    0.7500    0.3632      1200
           1     0.4609    0.4517    0.4562      1200
           2     0.3116    0.1633    0.2143      1200
           3     0.2474    0.2558    0.2515      1200
           4     0.4026    0.1050    0.1666      1200
           5     0.4653    0.0783    0.1341      1200
           6     0.4473    0.3008    0.3597      1200
           7     0.2681    0.6317    0.3765      1200
           8     0.5229    0.1808    0.2687      1200
           9     0.5442    0.2875    0.3762      1200

    accuracy                         0.3205     12000
   macro avg     0.3910    0.3205    0.2967     12000
weighted avg     0.3910    0.3205    0.2967     12000



**Fin del cuaderno 02.** No se realizan predicciones finales aquí; el modelo guardado será usado en el cuaderno 03.